In [46]:
from typing import TypedDict, List
from typing_extensions import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool
# from langchain.graphs import 
from dotenv import load_dotenv
import operator


In [47]:
# ---- Define State ----
class State(TypedDict):
    query: str  # The input query
    results: Annotated[str, operator.add]  # List to collect results from agents with Join operator
    best: str  # The final best result

load_dotenv()

True

In [48]:
# Define calculator tools
@tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

@tool
def subtract(a: float, b: float) -> float:
    """Subtract the second number from the first."""
    return a - b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """Divide the first number by the second."""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

tools = [add, subtract, multiply, divide]


# LLM + tool binding
llm = AzureChatOpenAI(model="gpt-4o-mini", temperature=0, api_version='2024-08-01-preview').bind_tools(tools)
judge_llm = AzureChatOpenAI(model="gpt-4o-mini", temperature=0, api_version='2024-08-01-preview')


In [55]:
# ---- Mapper Node (splits work into parallel sends) ----
def map_node(state: State):
    # Preserve the current state while yielding sends
    current_state = {"query": state["query"], "results": [], "best": ""}
    
    n = 3
    for _ in range(n):
        yield Send("agent", {"query": state["query"]})
        
    # Return the preserved state after yields
    print(current_state)
    return current_state

# ---- Agent Node ----
def agent_node(state: State) -> State:
    resp = llm.invoke(state["query"])
    # Return a single string so the Join operator can aggregate multiple agent responses
    print(resp.content)
    return {"results": resp.content}

# ---- Reduce Node (LLM Judge) ----
def reduce_node(state: State) -> State:
    candidates = state["results"]
    prompt = (
        "You are a judge. You will be given multiple candidate answers.\n\n"
        f"Question: {state['query']}\n\n"
        "Candidates:\n"
    )
    for i, c in enumerate(candidates, 1):
        prompt += f"{i}. {c}\n"
    prompt += "\nPick the single best candidate (just return the text of the best one)."

    judgment = judge_llm.invoke(prompt)
    return {"best": judgment.content}

In [56]:
workflow = StateGraph(State)

# ✅ mark map_node as a generator
workflow.add_node("map", map_node)
workflow.add_node("agent", agent_node)
workflow.add_node("reduce", reduce_node)

workflow.add_edge(START, "map")
workflow.add_edge("map", "agent")     # fan-out
workflow.add_edge("agent", "reduce")  # fan-in
workflow.add_edge("reduce", END)


app = workflow.compile()

In [57]:
final_state = app.invoke({"query": "calculate the expression (3 + 5)"})
print("\n=== Final Best Response ===")
print(final_state["best"])

{'query': 'calculate the expression (3 + 5)', 'results': [], 'best': ''}



=== Final Best Response ===
8


-----------------------------------------------------------------------------------------------------------------------------------------------------